# 05_embeddings_comparison.ipynb

**Actividad de Grado MIA UC**  
**Sistema Inteligente de Vigilancia de Riesgos Documentales basado en RAG y LLMs**

Este notebook genera embeddings sobre los chunks recursivos y compara dos modelos:

1. **Baseline:** `sentence-transformers/all-MiniLM-L6-v2`
2. **Mejorado:** `BAAI/bge-m3`

## Objetivo
Construir representaciones vectoriales del corpus para habilitar búsqueda semántica y preparar la evaluación de retrieval.

## Entradas
- `chunks_recursive.parquet`
- `gold_questions.csv`

## Salidas
- `embeddings_minilm.parquet`
- `embeddings_bge_m3.parquet`
- `retrieval_results_minilm.csv`
- `retrieval_results_bge_m3.csv`
- `resumen_embeddings_comparison.csv`


In [ ]:
# =====================================================
# 05_embeddings_comparison.ipynb
# Actividad de Grado MIA UC
# Patricia Patiño
# =====================================================

!pip -q install pandas pyarrow sentence-transformers scikit-learn openpyxl

## 1. Importar librerías

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import files
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 140)

print('Librerías cargadas correctamente')

## 2. Cargar archivos

Sube:
- `chunks_recursive.parquet`
- `gold_questions.csv`


In [ ]:
print('Sube chunks_recursive.parquet y gold_questions.csv')
uploaded = files.upload()
print('\nArchivos cargados:', list(uploaded.keys()))

## 3. Leer y validar datos

In [ ]:
chunks_path = Path('chunks_recursive.parquet')
questions_path = Path('gold_questions.csv')

if not chunks_path.exists():
    raise FileNotFoundError('No se encontró chunks_recursive.parquet')

if not questions_path.exists():
    raise FileNotFoundError('No se encontró gold_questions.csv')

chunks_df = pd.read_parquet(chunks_path)
gold_questions = pd.read_csv(questions_path)

required_chunks = {'doc_id', 'filename', 'tipo_documento', 'page', 'chunk_id', 'chunk_text'}
required_questions = {'question_id', 'question', 'categoria'}

missing_chunks = required_chunks - set(chunks_df.columns)
missing_questions = required_questions - set(gold_questions.columns)

if missing_chunks:
    raise ValueError(f'Faltan columnas en chunks: {missing_chunks}')

if missing_questions:
    raise ValueError(f'Faltan columnas en gold_questions: {missing_questions}')

chunks_df = chunks_df.reset_index(drop=True)
gold_questions = gold_questions.reset_index(drop=True)

print('Chunks cargados:', len(chunks_df))
print('Preguntas cargadas:', len(gold_questions))

display(chunks_df.head(3))
display(gold_questions.head(3))

## 4. Configuración de modelos

In [ ]:
MODELOS = {
    'minilm': 'sentence-transformers/all-MiniLM-L6-v2',
    'bge_m3': 'BAAI/bge-m3'
}

TEXT_COLUMN = 'chunk_text'
TOP_K = 5

print('Modelos a evaluar:')
for alias, model_name in MODELOS.items():
    print(f'- {alias}: {model_name}')

## 5. Funciones auxiliares

Estas funciones generan embeddings, ejecutan búsqueda semántica y devuelven los top-k chunks más similares para cada pregunta.

In [ ]:
def generar_embeddings(model, textos, batch_size=16):
    """Genera embeddings normalizados para una lista de textos."""
    embeddings = model.encode(
        textos,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    return embeddings


def buscar_top_k(question_embedding, chunk_embeddings, chunks_df, top_k=5):
    """Recupera los top-k chunks más similares a una pregunta."""
    scores = cosine_similarity(
        question_embedding.reshape(1, -1),
        chunk_embeddings
    )[0]

    top_indices = np.argsort(scores)[::-1][:top_k]

    resultados = []
    for rank, idx in enumerate(top_indices, start=1):
        row = chunks_df.iloc[idx]
        resultados.append({
            'rank': rank,
            'score': float(scores[idx]),
            'chunk_id': row['chunk_id'],
            'doc_id': row['doc_id'],
            'filename': row['filename'],
            'tipo_documento': row['tipo_documento'],
            'page': row['page'],
            'chunk_text': row['chunk_text']
        })

    return resultados


def evaluar_modelo(alias, model_name, chunks_df, gold_questions, top_k=5):
    """Genera embeddings de chunks y preguntas, y ejecuta recuperación top-k."""
    print('='*100)
    print(f'Cargando modelo: {model_name}')
    print('='*100)

    model = SentenceTransformer(model_name)

    textos_chunks = chunks_df[TEXT_COLUMN].astype(str).tolist()
    textos_preguntas = gold_questions['question'].astype(str).tolist()

    print('\nGenerando embeddings de chunks...')
    chunk_embeddings = generar_embeddings(model, textos_chunks)

    print('\nGenerando embeddings de preguntas...')
    question_embeddings = generar_embeddings(model, textos_preguntas)

    registros = []

    print('\nEjecutando recuperación semántica...')
    for i, qrow in gold_questions.iterrows():
        q_emb = question_embeddings[i]
        top_results = buscar_top_k(q_emb, chunk_embeddings, chunks_df, top_k=top_k)

        for result in top_results:
            registros.append({
                'model_alias': alias,
                'model_name': model_name,
                'question_id': qrow['question_id'],
                'question': qrow['question'],
                'categoria': qrow['categoria'],
                'rank': result['rank'],
                'score': result['score'],
                'chunk_id': result['chunk_id'],
                'doc_id': result['doc_id'],
                'filename': result['filename'],
                'tipo_documento': result['tipo_documento'],
                'page': result['page'],
                'chunk_text': result['chunk_text']
            })

    results_df = pd.DataFrame(registros)

    embeddings_df = chunks_df.copy()
    embeddings_df['embedding'] = [emb.tolist() for emb in chunk_embeddings]
    embeddings_df['embedding_model_alias'] = alias
    embeddings_df['embedding_model_name'] = model_name
    embeddings_df['embedding_dimension'] = chunk_embeddings.shape[1]

    resumen = {
        'model_alias': alias,
        'model_name': model_name,
        'num_chunks': len(chunks_df),
        'num_questions': len(gold_questions),
        'embedding_dimension': chunk_embeddings.shape[1],
        'top_k': top_k,
        'mean_top1_score': results_df[results_df['rank'] == 1]['score'].mean(),
        'mean_top5_score': results_df['score'].mean()
    }

    return embeddings_df, results_df, resumen

## 6. Ejecutar modelo baseline: MiniLM

Este modelo es liviano, rápido y sirve como baseline reproducible.

In [ ]:
embeddings_minilm_df, retrieval_minilm_df, resumen_minilm = evaluar_modelo(
    alias='minilm',
    model_name=MODELOS['minilm'],
    chunks_df=chunks_df,
    gold_questions=gold_questions,
    top_k=TOP_K
)

display(pd.DataFrame([resumen_minilm]))
display(retrieval_minilm_df.head())

## 7. Ejecutar modelo mejorado: BGE-M3

BGE-M3 es un modelo multilingüe más robusto para recuperación semántica. Puede tardar más que MiniLM.

In [ ]:
embeddings_bge_df, retrieval_bge_df, resumen_bge = evaluar_modelo(
    alias='bge_m3',
    model_name=MODELOS['bge_m3'],
    chunks_df=chunks_df,
    gold_questions=gold_questions,
    top_k=TOP_K
)

display(pd.DataFrame([resumen_bge]))
display(retrieval_bge_df.head())

## 8. Comparación inicial de modelos

Esta comparación todavía no mide respuesta correcta, porque para eso se necesita completar manualmente los documentos esperados.  
Por ahora compara dimensiones, similitud promedio y comportamiento de recuperación.

In [ ]:
resumen_comparison = pd.DataFrame([resumen_minilm, resumen_bge])
display(resumen_comparison)

retrieval_all_df = pd.concat([retrieval_minilm_df, retrieval_bge_df], ignore_index=True)

comparacion_scores = (
    retrieval_all_df.groupby(['model_alias', 'rank'])['score']
    .mean()
    .reset_index()
    .pivot(index='rank', columns='model_alias', values='score')
)

display(comparacion_scores)

## 9. Inspección manual de resultados

Revisa una pregunta específica para comparar qué recupera cada modelo.

In [ ]:
QUESTION_ID_TO_REVIEW = 'Q001'

for model_alias in ['minilm', 'bge_m3']:
    print('='*100)
    print('Modelo:', model_alias)
    print('='*100)

    subset = retrieval_all_df[
        (retrieval_all_df['question_id'] == QUESTION_ID_TO_REVIEW) &
        (retrieval_all_df['model_alias'] == model_alias)
    ].sort_values('rank')

    for _, row in subset.iterrows():
        print(f"Rank {row['rank']} | Score {row['score']:.4f} | {row['filename']} | {row['chunk_id']}")
        print(row['chunk_text'][:700])
        print('-'*100)

## 10. Preparar tabla para evaluación manual

Este archivo permitirá revisar si cada resultado recuperado es relevante o no.  
La columna `relevante_manual` queda vacía para llenarla después con 1 o 0.

In [ ]:
retrieval_manual_eval = retrieval_all_df.copy()
retrieval_manual_eval['relevante_manual'] = ''
retrieval_manual_eval['comentario_evaluador'] = ''

columnas_eval = [
    'model_alias', 'question_id', 'question', 'categoria', 'rank', 'score',
    'doc_id', 'filename', 'chunk_id', 'tipo_documento', 'page',
    'relevante_manual', 'comentario_evaluador', 'chunk_text'
]

retrieval_manual_eval = retrieval_manual_eval[columnas_eval]
display(retrieval_manual_eval.head())

## 11. Guardar artefactos

In [ ]:
embeddings_minilm_df.to_parquet('embeddings_minilm.parquet', index=False)
embeddings_bge_df.to_parquet('embeddings_bge_m3.parquet', index=False)

retrieval_minilm_df.to_csv('retrieval_results_minilm.csv', index=False, encoding='utf-8-sig')
retrieval_bge_df.to_csv('retrieval_results_bge_m3.csv', index=False, encoding='utf-8-sig')
retrieval_all_df.to_csv('retrieval_results_all_models.csv', index=False, encoding='utf-8-sig')
retrieval_manual_eval.to_excel('retrieval_manual_evaluation_template.xlsx', index=False)
resumen_comparison.to_csv('resumen_embeddings_comparison.csv', index=False, encoding='utf-8-sig')

print('Archivos generados:')
print('- embeddings_minilm.parquet')
print('- embeddings_bge_m3.parquet')
print('- retrieval_results_minilm.csv')
print('- retrieval_results_bge_m3.csv')
print('- retrieval_results_all_models.csv')
print('- retrieval_manual_evaluation_template.xlsx')
print('- resumen_embeddings_comparison.csv')

## 12. Descargar artefactos

In [ ]:
files.download('embeddings_minilm.parquet')
files.download('embeddings_bge_m3.parquet')
files.download('retrieval_results_minilm.csv')
files.download('retrieval_results_bge_m3.csv')
files.download('retrieval_results_all_models.csv')
files.download('retrieval_manual_evaluation_template.xlsx')
files.download('resumen_embeddings_comparison.csv')